# Laboratorio L5 - Pulire i dati

B.6 - Data science e Machine Learning: dai dati ai modelli

File necessario: `tragitti_esempio.csv` (oppure il CSV del questionario della classe: in questo caso adattare i nomi delle colonne e le correzioni).

Regola: il file originale non si modifica. Si lavora su una copia in memoria e alla fine si salva un nuovo file, `tragitti_puliti_classe.csv`. Ogni operazione va annotata nel registro della pulizia (ultima sezione).

In [ ]:
import pandas as pd

def controlla(condizione, suggerimento="Non ancora: rileggere la consegna."):
    """Stampa un riscontro immediato sull'esercizio."""
    print("Corretto." if condizione else suggerimento)

originale = pd.read_csv("tragitti_esempio.csv")
dati = originale.copy()       # copia di lavoro: l'originale resta intatto
print(dati.shape)
dati.head(10)

In [ ]:
dati.info()

## 1. Uniformare il testo: il mezzo di trasporto (esercizio 1)

- `str.strip()`: toglie gli spazi all'inizio e alla fine
- `str.lower()`: trasforma in minuscolo
- `replace(dizionario)`: sostituisce ogni valore che compare come chiave del dizionario con il valore associato

Un dizionario Python si scrive tra graffe: `{"vecchio": "nuovo", ...}`.

In [ ]:
dati["mezzo"] = dati["mezzo"].str.strip().str.lower()
dati["mezzo"].value_counts()

Completare il dizionario `correzioni` in modo che restino solo i sei mezzi del dizionario dei dati: `piedi`, `bici`, `monopattino`, `bus`, `auto`, `treno`. Per `treno + bus` si applica la regola del questionario: conta il mezzo su cui si passa più tempo (il treno).

In [ ]:
correzioni = {
    "autobus": "bus",
    "pullman": "bus",
    "a piedi": "piedi",
    "bicicletta": "bici",
    "macchina": ___,
    "in auto": ___,
    # aggiungere le correzioni mancanti
    "treno + bus": "treno",
}
dati["mezzo"] = dati["mezzo"].replace(correzioni)
print(dati["mezzo"].value_counts())
controlla(set(dati["mezzo"].unique()) == {"piedi", "bici", "monopattino", "bus", "auto", "treno"},
          "Controllare quali valori compaiono ancora oltre ai sei ammessi.")

## 2. Numeri scritti come testo (esercizio 2)

Distanza: alcune risposte usano la virgola decimale (`1,2`).

- `astype(str)`: tratta ogni valore come testo
- `str.replace(",", ".")`: sostituisce la virgola con il punto
- `pd.to_numeric(..., errors="coerce")`: converte in numero; ciò che non è convertibile diventa `NaN`

In [ ]:
dati["distanza_km"] = pd.to_numeric(dati["distanza_km"].astype(str).str.replace(",", "."), errors="coerce")
print(dati["distanza_km"].dtype, "- valori mancanti:", dati["distanza_km"].isna().sum())

Tempo: alcune risposte contengono l'unità (`45 min`, `45 minuti`, `45'`). Si estrae la parte numerica.

- `str.extract(r"(\d+)")`: estrae la prima sequenza di cifre del testo; `\d` indica una cifra, `+` una o più, le parentesi indicano la parte da estrarre. È un'espressione regolare, un linguaggio per descrivere schemi di testo
- `[0]`: `extract` restituisce una tabella; si prende la sua prima (e unica) colonna

In [ ]:
print("prima:", dati["tempo_min"].astype(str).str.contains("[a-z']", regex=True).sum(), "risposte con testo")
dati["tempo_min"] = pd.to_numeric(dati["tempo_min"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
print(dati["tempo_min"].dtype, "- valori mancanti:", dati["tempo_min"].isna().sum())
controlla(dati["tempo_min"].dtype.kind == "f" and dati["distanza_km"].dtype.kind == "f")

## 3. Duplicati (esercizio 3)

Un modulo online può ricevere due volte la stessa risposta (doppio clic, invio ripetuto). Le risposte duplicate hanno le stesse risposte ma data di invio e `id` diversi, quindi il confronto va fatto solo sulle colonne delle risposte (`subset`).

- `duplicated(subset=[...])`: `True` per ogni riga uguale a una riga precedente nelle colonne indicate
- `keep=False`: segna tutte le copie, anche la prima, per poterle esaminare
- `drop_duplicates(subset=[...])`: elimina le copie successive alla prima

In [ ]:
risposte = ["anno_corso", "mezzo", "distanza_km", "tempo_min", "fascia_partenza"]
print("righe duplicate:", dati.duplicated(subset=risposte).sum())
dati[dati.duplicated(subset=risposte, keep=False)].sort_values("distanza_km")

**Domanda**: due studenti diversi potrebbero dare esattamente le stesse risposte? Con quali colonne del dataset lo si potrebbe verificare? Qui si considerano duplicati.

Risposta:

In [ ]:
dati = dati.___(subset=risposte)
print(dati.shape)
controlla(len(dati) == 150)

## 4. Valori impossibili e valori anomali (esercizio 4)

- valori impossibili: violano le regole del dizionario dei dati (distanza negativa o zero, tempo zero); sono sicuramente errori
- valori anomali (outlier): molto lontani dagli altri; possono essere errori o casi reali

Criterio dei quartili: è anomalo un valore minore di Q1 - 1,5 x IQR o maggiore di Q3 + 1,5 x IQR (lo stesso usato dai baffi del boxplot).

In [ ]:
print(dati[(dati["distanza_km"] <= 0) | (dati["tempo_min"] <= 0)][["id", "mezzo", "distanza_km", "tempo_min"]])

In [ ]:
def anomali(colonna):
    q1, q3 = colonna.quantile(0.25), colonna.quantile(0.75)
    iqr = q3 - q1
    return (colonna < q1 - 1.5 * iqr) | (colonna > q3 + 1.5 * iqr)

dati[anomali(dati["distanza_km"])][["id", "mezzo", "distanza_km", "tempo_min"]].sort_values("distanza_km")

**Domanda**: tra le distanze segnalate come anomale, quali sono probabilmente errori e quali casi reali? Guardare il mezzo e il tempo di ciascuna.

Risposta:

Decisione: i valori impossibili e i valori anomali giudicati errori vengono sostituiti con `NaN` (valore mancante), non corretti a mano: 250 km potrebbe essere 2,50 km, ma non è possibile saperlo. I casi reali restano.

- `dati.loc[condizione, "colonna"] = valore`: assegna il valore alla colonna solo nelle righe che soddisfano la condizione

In [ ]:
import numpy as np

dati.loc[dati["distanza_km"] <= 0, "distanza_km"] = np.nan
dati.loc[dati["distanza_km"] > 100, "distanza_km"] = np.nan
dati.loc[dati["tempo_min"] <= 0, "tempo_min"] = np.nan
dati.loc[dati["tempo_min"] > ___, "tempo_min"] = np.nan
print(dati[["distanza_km", "tempo_min"]].describe().round(1))
controlla(dati["tempo_min"].max() <= 150 and dati["distanza_km"].min() > 0)

## 5. Valori mancanti (esercizio 5)

- `isna().sum()`: numero di valori mancanti per colonna
- `dropna(subset=[...])`: elimina le righe con valori mancanti nelle colonne indicate
- `fillna(valore)`: sostituisce i valori mancanti con un valore (per esempio la mediana)

Eliminare riduce il campione; sostituire inventa un valore plausibile ma non vero. La scelta dipende dall'uso: per un grafico della distanza bastano le righe con la distanza; per un modello servono righe complete.

In [ ]:
print(dati.isna().sum())

In [ ]:
completi = dati.dropna(subset=["distanza_km", "tempo_min"])
print("righe complete per distanza e tempo:", len(completi), "su", len(dati))

## 6. Salvare e documentare

- `to_csv(nome, index=False)`: salva il DataFrame in un nuovo CSV senza la colonna dell'indice
- la colonna `inviato_il` non serve più all'analisi e si elimina (minimizzazione)

In [ ]:
pulito = dati.drop(columns=["inviato_il"])
pulito.to_csv("tragitti_puliti_classe.csv", index=False)
print(pulito.shape)
pulito.head()

### Registro della pulizia

Completare con le operazioni svolte, il numero di righe o valori coinvolti e la motivazione.

| operazione | righe o valori coinvolti | motivazione |
|---|---|---|
| mezzo: spazi, minuscole, varianti | | |
| distanza: virgola decimale | | |
| tempo: unità nel testo | | |
| duplicati eliminati | | |
| valori impossibili resi mancanti | | |
| valori anomali resi mancanti | | |
| colonna eliminata | | |